In [52]:
import openmeteo_requests
import eurostat
import requests_cache
import pandas as pd
from retry_requests import retry

In [53]:
# Weather Data
# Ensures openmeteo has not been executed already.
try:
    openmeteo
except NameError:
    cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
    retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
    openmeteo = openmeteo_requests.Client(session = retry_session)

# API URL
url = 'https://archive-api.open-meteo.com/v1/archive'
# API parameters
parameters = {
    'latitude': 35.9167,
    'longitude': 14.4333,
    'start_date': '2010-01-01',
    'end_date': '2025-12-31',
    'daily': ['temperature_2m_mean', 'temperature_2m_max', 'apparent_temperature_mean', 'sunshine_duration', 'daylight_duration', 'precipitation_sum', 'wind_speed_10m_max', 'weather_code'],
    'timezone': 'Europe/Berlin',
}

# Retrieval of API responses
responses = openmeteo.weather_api(url, params=parameters)
# Checks if responses were received
if responses:
    response = responses[0]
    daily = response.Daily()

    # Data pulled from response
    weather_data = {
        'mean_temp': daily.Variables(0).ValuesAsNumpy(),
        'max_temp': daily.Variables(1).ValuesAsNumpy(),
        'feels_like': daily.Variables(2).ValuesAsNumpy(),
        'sunshine_dur': daily.Variables(3).ValuesAsNumpy(),
        'daylight_dur': daily.Variables(4).ValuesAsNumpy(),
        'precipitation': daily.Variables(5).ValuesAsNumpy(),
        'wind_speed': daily.Variables(6).ValuesAsNumpy(),
        'weather_code': daily.Variables(7).ValuesAsNumpy()
    }

    # Data fixed into dataframe
    df_weather_raw = pd.DataFrame(weather_data)
    print(df_weather_raw)
else:
    print('No data retrieved from weather api.')


      mean_temp   max_temp  feels_like  sunshine_dur  daylight_dur  \
0     17.038748  18.507500   12.691882  34635.476562  35185.632812   
1     15.326251  16.007500    7.708334  34146.585938  35221.769531   
2     14.495000  15.107500    8.478354  33864.101562  35260.875000   
3     14.947083  16.807501   11.940295  27727.416016  35302.890625   
4     16.984583  18.307501   13.001320  30295.017578  35347.800781   
...         ...        ...         ...           ...           ...   
5839  15.460418  16.700001   12.828908  32277.310547  35051.968750   
5840  14.456249  17.100000   13.398805  33581.242188  35072.425781   
5841  13.739586  16.350000   13.159008  29120.347656  35096.046875   
5842  13.622916  15.400000   11.149653  33344.414062  35122.812500   
5843  13.218750  15.400000   10.697787  32398.789062  35152.710938   

      precipitation  wind_speed  weather_code  
0               0.0   43.601631           2.0  
1               0.2   54.553955          51.0  
2              

In [55]:
# Tourist data
# Retrieval of data using eurostat
df_tourism_raw = eurostat.get_data_df('tour_occ_arm')

# Checks if df_tourism_raw was retrieved
if df_tourism_raw is not None:
    # Mask to filter the Eurostat dataframe to only contain Maltese data
    mask = (df_tourism_raw['geo\\TIME_PERIOD'] == 'MT') & (df_tourism_raw['c_resid'] == 'TOTAL') & (df_tourism_raw['unit'] == 'NR') & (df_tourism_raw['nace_r2'] == 'I551-I553')
    df_tourism_raw = df_tourism_raw[mask]
    
print(df_tourism_raw)

     freq c_resid unit    nace_r2 geo\TIME_PERIOD  1990-01  1990-02  1990-03  \
1469    M   TOTAL   NR  I551-I553              MT      NaN      NaN      NaN   

      1990-04  1990-05  ...   2025-03   2025-04   2025-05   2025-06   2025-07  \
1469      NaN      NaN  ...  227983.0  249503.0  258737.0  252970.0  265532.0   

       2025-08   2025-09   2025-10   2025-11   2025-12  
1469  271574.0  245468.0  283159.0  225437.0  205171.0  

[1 rows x 437 columns]


In [ ]:
# Weather Data Cleaning
# Copy of raw dataframe
df_weather = df_weather_raw.copy()

# Sets date index to dataframe
dates = pd.date_range(start="2010-01-01", end="2025-12-31", freq='D')
df_weather.index = dates

# Converts data from daily to monthly using resample to MS (Month Start) and aggregate to use specific summary methodology
df_weather = df_weather.resample('MS').agg({
    'mean_temp': 'mean',
    'max_temp': 'max',
    'feels_like': 'mean',
    'sunshine_dur': 'sum',
    'daylight_dur': 'sum',
    'precipitation': 'sum',
    'wind_speed': 'mean',
    'weather_code': lambda x: x.mode().iloc[0] # As mode is not a set pandas summary methodology (also takes ties into account)
}).astype(float).round(1)

df_weather.index = df_weather.index.to_period('M')

# Weather code (DMO) mapping
wmo_description = {
    0: 'Clear sky',
    1: 'Mainly clear', 2: 'Partly cloudy', 3: 'Overcast',
    45: 'Fog', 48: 'Depositing rime fog',
    51: 'Drizzle: Light', 53: 'Drizzle: Moderate', 55: 'Drizzle: Dense',
    61: 'Rain: Slight', 63: 'Rain: Moderate', 65: 'Rain: Heavy',
    80: 'Rain showers: Slight', 81: 'Rain showers: Moderate', 82: 'Rain showers: Violent',
    95: 'Thunderstorm', 96: 'Thunderstorm with slight hail', 99: 'Thunderstorm with heavy hail'
}

df_weather['weather_desc'] = df_weather['weather_code'].map(wmo_description)

print(df_weather)